In [1]:
import xarray as xr
import torch
import sys
sys.path.append('../src')
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
import os
import numpy as np

In [2]:
data_dir = "/pscratch/sd/s/suryad/data"
wet_file = "3D_wet.pt"
surface_wet_file = "surface_wet.pt"
data_zarr = "3D_data"
data_means_zarr = "3D_data_means"
data_stds_zarr = "3D_data_stds"
grid_file = "Grid_New.nc"

inputs = INPT_VARS["3D_all"]
extra_in = EXTRA_VARS["3D_all"]
outputs = OUT_VARS["3D_all"]
lag = 1
interval = 1
hist = 1
N_samples = 20
N_val = 10
steps = 4


wet = torch.load(
    os.path.join(data_dir, wet_file)
)
data = xr.open_zarr(
    os.path.join(data_dir, data_zarr)
)
data_mean = xr.open_zarr(
    os.path.join(data_dir, data_means_zarr)
)
data_std = xr.open_zarr(
    os.path.join(data_dir, data_stds_zarr)
)

s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val
    


### Main

In [ ]:
class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        ind_start,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.ind_start = ind_start

        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        
        if self.hist > 0:
            total_steps = self.hist + 1
            rolling_dataset = data.rolling(time=len(data.time)-total_steps, center=False).construct('window_dim')
            rolling_dataset = rolling_dataset.isel(time=slice(len(data.time)-total_steps-1,None))
            # Long Assertion to make sure the rolling dataset is correct
            # for t in range(len(data.time)-self.hist):
            #     assert np.equal(data.isel(time=slice(t, t+self.hist)), rolling_dataset.isel(window_dim=t)).all()
            self.rolling_dataset = rolling_dataset
            self.rolling_inputs_without_extra = rolling_dataset[inputs_str]
            self.rolling_extra_inputs = rolling_dataset[extra_in_str]
            self.rolling_outputs = rolling_dataset[outputs_str]
            self.in_no_extra_mean = data_mean[inputs_str]
            self.in_no_extra_std = data_std[inputs_str]
            self.extra_mean = data_mean[extra_in_str]
            self.extra_std = data_std[extra_in_str]
            

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        if type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)

            # ind_in = slice(
            #     self.ind_start + self.hist + idx.start, self.ind_start + self.hist + idx.stop * self.interval, self.interval
            # )
            # ind_hist = [
            #     [self.ind_start + j + i for i in range(self.hist)]
            #     for j in range(idx.start, idx.stop)
            # ]
            ind_out = slice(
                self.ind_start + idx.start + self.hist + self.lag,
                self.ind_start + idx.stop * self.interval + self.hist + self.lag,
                self.interval,
            )
            
            # print("Index in: ", ind_in)
            # print("Hist: ", self.rolling_dataset.isel(window_dim=idx))
            print("Index out: ", ind_out)
            
            # print(self.rolling_dataset.isel(window_dim=idx).isel(time=-1).to_array().shape)
            # print(self.outputs.isel(time=ind_out).to_array().shape)
            
            data_in = self.rolling_inputs_without_extra.isel(window_dim=idx, time=slice(None, -1))
            data_in = ((data_in - self.in_no_extra_mean) / self.in_no_extra_std).fillna(0)
            data_in = data_in.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
            print(data_in)
            data_in = data_in.reshape((data_in.shape[0], -1, data_in.shape[3], data_in.shape[4]))
            print(data_in)
            data_in_boundary = self.rolling_extra_inputs.isel(window_dim=idx).isel(time=-2)
            data_in_boundary = ((data_in_boundary - self.extra_mean) / self.extra_std).fillna(0)
            data_in_boundary = data_in_boundary.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()
            data_in = np.concatenate((data_in, data_in_boundary), axis=1)
            

            label = self.rolling_outputs.isel(window_dim=idx).isel(time=-1)
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = label.to_array().transpose("time", "variable", "y", "x").to_numpy()

        elif type(idx) == int:
            assert idx >= 0
            ind_in = self.ind_start + idx * self.interval + self.hist
            ind_hist = [self.ind_start + idx * self.interval + i for i in range(self.hist)]
            ind_out = self.ind_start + idx * self.interval + self.hist + self.lag
            
            print("Index hist:", ind_hist)
            print("Index in: ", ind_in)
            print("Index out: ", ind_out)
                
            data_in = self.inputs.isel(time=ind_in)
            data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
            data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()

            if self.hist > 0:
                data_hist = self.outputs.isel(time=ind_hist)
                data_hist = ((data_hist - self.out_mean) / self.out_std).fillna(0) 
                data_hist = data_hist.to_array().transpose("time", "variable", "y", "x").to_numpy()
                data_hist = data_hist.reshape((-1, data_hist.shape[2], data_hist.shape[3]))
                data_in = np.concatenate((data_hist, data_in), axis=0)
            
            label = self.outputs.isel(time=ind_out)
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = label.to_array().transpose("variable", "y", "x").to_numpy()

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    device="cuda",
)

In [6]:
%%time
d = val_data[1:2]
print(d[0].shape)
print(d[1].shape)

Index out:  slice(24, 25, 1)
<xarray.DataArray (window_dim: 1, time: 2, variable: 77, y: 180, x: 360)>
dask.array<transpose, shape=(1, 2, 77, 180, 360), dtype=float64, chunksize=(1, 1, 1, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * time      (time) object 2022-12-19 12:00:00 2022-12-24 12:00:00
  * x         (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y         (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
  * variable  (variable) object 'uo_lev_0' 'uo_lev_1' ... 'so_lev_18' 'zos'
Dimensions without coordinates: window_dim


KeyboardInterrupt: 

In [13]:

class data_CNN_Disk_steps(torch.utils.data.Dataset):
    
    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.steps = steps
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        outputs = []
        for step in range(self.steps):
            if type(idx) == list:
                assert self.hist == 0
                ind_in = [i * self.interval + self.lag * step for i in idx]
                ind_out = [i * self.interval + self.lag * (step + 1) for i in idx]

            elif type(idx) == slice:
                assert self.hist == 0
                if idx.start == None and idx.stop == None:
                    idx = slice(0, self.size, idx.step)
                elif idx.start == None:
                    idx = slice(0, idx.stop, idx.step)
                elif idx.stop == None:
                    idx = slice(idx.start, self.size, idx.step)

                ind_in = slice(
                    idx.start, idx.stop * self.interval + self.lag * step, self.interval
                )
                ind_out = slice(
                    idx.start + self.lag,
                    idx.stop * self.interval + self.lag * (step + 1),
                    self.interval,
                )

            if type(idx) == int:
                assert idx >= 0 
                ind_in = idx * self.interval + self.lag * step
                ind_out = idx * self.interval + self.lag * (step + 1)
                
                # HIST
                # ind_in = [
                #     idx * self.interval + i + self.lag * step
                #     for i in range(self.hist + 1)
                # ]
                # ind_out = idx * self.interval + self.lag * (step + 1) + self.hist

            print("Index in: ", ind_in)
            print("Index out: ", ind_out)
            
            data_in = self.inputs.isel(time=ind_in)
            data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
            label = self.outputs.isel(time=ind_out)
            label = ((label - self.out_mean) / self.out_std).fillna(0)

            if type(idx) == int:
                if self.hist == 0:
                    data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()
                else:
                    data_in = data_in.to_array().transpose("time", "variable", "y", "x").to_numpy()
                    data_in = data_in.reshape((-1, data_in.shape[2], data_in.shape[3]))
                label = label.to_array().transpose("variable", "y", "x").to_numpy()
            else:
                assert self.hist == 0
                data_in = (
                    data_in.to_array()
                    .transpose("time", "variable", "y", "x")
                    .to_numpy()
                )
                label = (
                    label.to_array().transpose("time", "variable", "y", "x").to_numpy()
                )

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs


train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)

IndexError: arrays used as indices must be of integer (or boolean) type

In [106]:
d = train_data[0]
print(d[0].shape)
print(d[1].shape)

torch.Size([160, 180, 360])
torch.Size([77, 180, 360])


### Testing

In [22]:
# Create xarray dataset of size - (50, 60, 30, 30)
data = xr.DataArray(
    np.random.rand(50, 60, 30, 30),
    dims=["time", "variable", "y", "x"],
    coords={
        "time": np.arange(50),
        "variable": np.arange(60),
        "y": np.arange(30),
        "x": np.arange(30),
    },
)

data_without_boundary = data.isel(variable=slice(0, 50))
boundary = data.isel(variable=slice(50, 60))



In [23]:
# Create rolling indices of timesteps 
indices = xr.DataArray(
    np.arange(data.time.size),
    dims=["time"],
    coords={"time": data.time},
)
total_steps = hist + 1
# convert to long format
rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None))
rolling_indices
                                    

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


<xarray.DataArray (window_dim: 48, time: 3)>
array([[ 0,  1,  2],
       [ 1,  2,  3],
       [ 2,  3,  4],
       [ 3,  4,  5],
       [ 4,  5,  6],
       [ 5,  6,  7],
       [ 6,  7,  8],
       [ 7,  8,  9],
       [ 8,  9, 10],
       [ 9, 10, 11],
       [10, 11, 12],
       [11, 12, 13],
       [12, 13, 14],
       [13, 14, 15],
       [14, 15, 16],
       [15, 16, 17],
       [16, 17, 18],
       [17, 18, 19],
       [18, 19, 20],
       [19, 20, 21],
...
       [28, 29, 30],
       [29, 30, 31],
       [30, 31, 32],
       [31, 32, 33],
       [32, 33, 34],
       [33, 34, 35],
       [34, 35, 36],
       [35, 36, 37],
       [36, 37, 38],
       [37, 38, 39],
       [38, 39, 40],
       [39, 40, 41],
       [40, 41, 42],
       [41, 42, 43],
       [42, 43, 44],
       [43, 44, 45],
       [44, 45, 46],
       [45, 46, 47],
       [46, 47, 48],
       [47, 48, 49]])
Coordinates:
  * time     (time) int64 47 48 49
Dimensions without coordinates: window_dim

In [10]:
# Create rolling dataset itself
rolling_data = data.rolling(time=len(data.time)-total_steps, center=False).construct('window_dim')
data2 = rolling_data.transpose('window_dim', 'time', 'variable', 'y', 'x').isel(time=slice(len(data.time)-total_steps-1,None))

In [11]:
# np.equal(data2.isel(window_dim=10), data.isel(time=slice(10, hist+10))).all()

In [12]:
rolling_indices

<xarray.DataArray (window_dim: 48, time: 3)>
array([[ 0,  1,  2],
       [ 1,  2,  3],
       [ 2,  3,  4],
       [ 3,  4,  5],
       [ 4,  5,  6],
       [ 5,  6,  7],
       [ 6,  7,  8],
       [ 7,  8,  9],
       [ 8,  9, 10],
       [ 9, 10, 11],
       [10, 11, 12],
       [11, 12, 13],
       [12, 13, 14],
       [13, 14, 15],
       [14, 15, 16],
       [15, 16, 17],
       [16, 17, 18],
       [17, 18, 19],
       [18, 19, 20],
       [19, 20, 21],
...
       [28, 29, 30],
       [29, 30, 31],
       [30, 31, 32],
       [31, 32, 33],
       [32, 33, 34],
       [33, 34, 35],
       [34, 35, 36],
       [35, 36, 37],
       [36, 37, 38],
       [37, 38, 39],
       [38, 39, 40],
       [39, 40, 41],
       [40, 41, 42],
       [41, 42, 43],
       [42, 43, 44],
       [43, 44, 45],
       [44, 45, 46],
       [45, 46, 47],
       [46, 47, 48],
       [47, 48, 49]])
Coordinates:
  * time     (time) int64 47 48 49
Dimensions without coordinates: window_dim

In [13]:
# Use rolling indices to get the rolling dataset
data3 = data.isel(time=rolling_indices.isel(window_dim=slice(None,2)).values)

IndexError: Unlabeled multi-dimensional array cannot be used for indexing: time

<xarray.DataArray (window_dim: 3, time: 2, variable: 6)>
array([[[0.33547378, 0.67330893, 0.69904389, 0.88787631, 0.26807342,
         0.07760665],
        [0.78355031, 0.6135081 , 0.75868513, 0.16590802, 0.71739294,
         0.42383822]],

       [[0.78355031, 0.6135081 , 0.75868513, 0.16590802, 0.71739294,
         0.42383822],
        [0.01768034, 0.5773279 , 0.09635795, 0.0637734 , 0.63216361,
         0.78761642]],

       [[0.01768034, 0.5773279 , 0.09635795, 0.0637734 , 0.63216361,
         0.78761642],
        [0.4377235 , 0.42413106, 0.16612197, 0.1085243 , 0.35388582,
         0.47942606]]])
Coordinates:
  * time      (time) int64 0 1
  * variable  (variable) int64 0 1 2 3 4 5
Dimensions without coordinates: window_dim


<xarray.DataArray (window_dim: 3, time: 2, variable: 6)>
array([[[0.33547378, 0.67330893, 0.69904389, 0.88787631, 0.26807342,
         0.07760665],
        [0.78355031, 0.6135081 , 0.75868513, 0.16590802, 0.71739294,
         0.42383822]],

       [[0.78355031, 0.6135081 , 0.75868513, 0.16590802, 0.71739294,
         0.42383822],
        [0.01768034, 0.5773279 , 0.09635795, 0.0637734 , 0.63216361,
         0.78761642]],

       [[0.01768034, 0.5773279 , 0.09635795, 0.0637734 , 0.63216361,
         0.78761642],
        [0.4377235 , 0.42413106, 0.16612197, 0.1085243 , 0.35388582,
         0.47942606]]])
Coordinates:
  * time      (time) int64 0 1
  * variable  (variable) int64 0 1 2 3 4 5
Dimensions without coordinates: window_dim


In [24]:
# Create dummy dataset
data = xr.DataArray(
    np.random.rand(10, 6),
    dims=["time", "variable"],
    coords={
        "time": np.arange(10),
        "variable": np.arange(6),
    },
)
data = data.expand_dims("window_dim", axis=0)
slices = [slice(0, 2), slice(1, 3), slice(2, 4)]
xr.concat([data.isel(time=sl) for sl in slices], dim="window_dim")

<xarray.DataArray (window_dim: 3, time: 4, variable: 6)>
array([[[0.75722179, 0.04417218, 0.6719664 , 0.08574452, 0.60713179,
         0.3262955 ],
        [0.75094376, 0.51528978, 0.27224893, 0.72990428, 0.88792956,
         0.78279852],
        [       nan,        nan,        nan,        nan,        nan,
                nan],
        [       nan,        nan,        nan,        nan,        nan,
                nan]],

       [[       nan,        nan,        nan,        nan,        nan,
                nan],
        [0.75094376, 0.51528978, 0.27224893, 0.72990428, 0.88792956,
         0.78279852],
        [0.85588419, 0.37050595, 0.4137549 , 0.7152108 , 0.02981982,
         0.07150118],
        [       nan,        nan,        nan,        nan,        nan,
                nan]],

       [[       nan,        nan,        nan,        nan,        nan,
                nan],
        [       nan,        nan,        nan,        nan,        nan,
                nan],
        [0.85588419, 0.37050595, 0.4137549 , 0.7152108 , 0.02981982,
         0.07150118],
        [0.59335483, 0.24247981, 0.07688977, 0.02362617, 0.40318536,
         0.43106402]]])
Coordinates:
  * time      (time) int64 0 1 2 3
  * variable  (variable) int64 0 1 2 3 4 5
Dimensions without coordinates: window_dim

In [15]:
data = np.random.rand(20, 5, 2, 2)  # Shape: (time, variable, y, x)
dims = ('time', 'variable', 'y', 'x')
coords = {'time': np.arange(20), 'variable': np.arange(5), 'y': np.arange(2), 'x': np.arange(2)}
dataset = xr.DataArray(data, dims=dims, coords=coords)

datamean = dataset.mean(['time', 'y', 'x'])
datastd = dataset.std(['time', 'y', 'x'])

# Apply rolling window along the 'time' dimension with a window size of 3
steps = 1
rolling_dataset = dataset.rolling(time=len(dataset.time)-steps, center=False).construct('window_dim')
dataset2 = rolling_dataset.transpose('window_dim', 'time', 'variable', 'y', 'x').isel(time=slice(len(dataset.time)-steps-1,None))

dataset2

<xarray.DataArray (window_dim: 19, time: 2, variable: 5, y: 2, x: 2)>
array([[[[[0.95044234, 0.61617874],
          [0.74625926, 0.22862026]],

         [[0.11962574, 0.53522423],
          [0.56518212, 0.64985218]],

         [[0.65813133, 0.37163395],
          [0.86048459, 0.10605899]],

         [[0.91840213, 0.20074558],
          [0.19197724, 0.65301834]],

         [[0.18209919, 0.19680095],
          [0.41782647, 0.39356398]]],


        [[[0.76547966, 0.71167524],
          [0.81526788, 0.43023282]],

         [[0.05772425, 0.30142496],
...
          [0.21854556, 0.21343278]],

         [[0.26744615, 0.10458113],
          [0.33767942, 0.03734453]]],


        [[[0.88088439, 0.54643488],
          [0.58909968, 0.85980792]],

         [[0.69228839, 0.21967251],
          [0.42570267, 0.77763655]],

         [[0.89428669, 0.64185989],
          [0.46246056, 0.21641632]],

         [[0.76334172, 0.65394288],
          [0.51963464, 0.1381153 ]],

         [[0.4844287 , 0.67599708],
          [0.8570143 , 0.26383563]]]]])
Coordinates:
  * time      (time) int64 18 19
  * variable  (variable) int64 0 1 2 3 4
  * y         (y) int64 0 1
  * x         (x) int64 0 1
Dimensions without coordinates: window_dim

In [16]:
# Assertions
for t in range(len(dataset.time)-steps):
    assert np.equal(dataset.sel(time=slice(t, t+steps)).data, dataset2.isel(window_dim=t).data).all()

In [17]:
boundary = dataset2.isel(window_dim=0).isel(variable=slice(None,2))

In [18]:
inputs = dataset2.isel(window_dim=0).isel(variable=slice(2,None))

In [19]:
inputs

<xarray.DataArray (time: 2, variable: 3, y: 2, x: 2)>
array([[[[0.65813133, 0.37163395],
         [0.86048459, 0.10605899]],

        [[0.91840213, 0.20074558],
         [0.19197724, 0.65301834]],

        [[0.18209919, 0.19680095],
         [0.41782647, 0.39356398]]],


       [[[0.60505689, 0.68267901],
         [0.13203332, 0.51283844]],

        [[0.62768799, 0.14238029],
         [0.04247375, 0.03324253]],

        [[0.74735502, 0.08945167],
         [0.08277978, 0.85642097]]]])
Coordinates:
  * time      (time) int64 18 19
  * variable  (variable) int64 2 3 4
  * y         (y) int64 0 1
  * x         (x) int64 0 1

In [20]:
boundary

<xarray.DataArray (time: 2, variable: 2, y: 2, x: 2)>
array([[[[0.95044234, 0.61617874],
         [0.74625926, 0.22862026]],

        [[0.11962574, 0.53522423],
         [0.56518212, 0.64985218]]],


       [[[0.76547966, 0.71167524],
         [0.81526788, 0.43023282]],

        [[0.05772425, 0.30142496],
         [0.96519693, 0.51068366]]]])
Coordinates:
  * time      (time) int64 18 19
  * variable  (variable) int64 0 1
  * y         (y) int64 0 1
  * x         (x) int64 0 1